In [1]:
from huggingface_hub import hf_hub_download
import importlib.util

repo = "Horizon-Labs/punctuation-restoration-small"

script_path = hf_hub_download(
    repo_id=repo,
    filename="punctuate.py"
)

spec = importlib.util.spec_from_file_location(
    "punctuate", script_path
)

module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

punctuator = module.Punctuator(repo)

text = """
bugun havo juda yaxshi ertaga esa yomgir yogishi mumkin
"""

restored = punctuator(text)

print(restored)

/Users/macbook/Documents/IT/projects/tts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


bugun havo juda yaxshi, ertaga esa yomgir yogishi mumkin.


In [3]:
text_2 = """kecha ertalab quyosh hali toʻliq koʻtarilmasdan oldin shahar koʻchalarida odamlar ishga va oʻqishga shoshilib ketayotgan bir paytda men qahva ichib oʻtirgan edim va ertangi rejam haqida chuqur oʻylab koʻrdim
agar siz har kuni kamida yarim soat kitob oʻqisangiz va yangi soʻzlarni kundalik hayotingizda faol qoʻllashga harakat qilsangiz bir necha oydan keyin nutqingiz ancha ravon va boy boʻlib qolishiga shubha qilmayman
bahor fasli kirib kelishi bilan tabiat uygʻonib daraxtlar kurtak chiqara boshlaganda bogʻdagi qushlarning sayrashi va shamolning mayin esishi odamga oʻzgacha ilhom va xotirjamlik bagʻishlaydi
biz oʻtgan yili yozda doʻstlarimiz bilan togʻ etagiga dam olishga borganimizda kechqurun gulxan yoqib atrofida oʻtirdik va bolalik xotiralari hamda kelajakdagi orzularimiz haqida uzoq suhbatlashdik
zamonaviy texnologiyalar kundalik hayotimizni sezilarli darajada osonlashtirgan boʻlsa-da jonli muloqot va tabiat qoʻynida vaqt oʻtkazish insonning ruhiy salomatligi uchun hamon eng muhim omillardan biri boʻlib qolmoqda"""

restored_2 = punctuator(text_2)

print(restored_2)

kecha ertalab quyosh hali toʻliq koʻtarilmasdan oldin shahar koʻchalarida odamlar ishga va oʻqishga shoshilib ketayotgan bir paytda men qahva ichib oʻtirgan edim va ertangi rejam haqida chuqur oʻylab koʻrdim. agar siz har kuni kamida yarim soat kitob oʻqisangiz va yangi soʻzlarni kundalik hayotingizda faol qoʻllashga harakat qilsangiz, bir necha oydan keyin nutqingiz ancha ravon va boy boʻlib qolishiga shubha qilmayman. bahor fasli kirib kelishi bilan tabiat uygʻonib, daraxtlar kurtak chiqara boshlaganda, bogʻdagi qushlarning sayrashi va shamolning mayin esishi odamga oʻzgacha ilhom va xotirjamlik bagʻishlaydi. biz oʻtgan yili yozda doʻstlarimiz bilan togʻ etagiga dam olishga borganimizda kechqurun gulxan yoqib, atrofida oʻtirdik va bolalik xotiralari hamda kelajakdagi orzularimiz haqida uzoq suhbatlashdik. zamonaviy texnologiyalar kundalik hayotimizni sezilarli darajada osonlashtirgan boʻlsa-da, jonli muloqot va tabiat qoʻynida vaqt oʻtkazish insonning ruhiy salomatligi uchun hamon en

In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_path = "islomov/rubai-corrector-transcript-uz"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

text = """kecha ertalab quyosh hali toʻliq koʻtarilmasdan oldin shahar koʻchalarida odamlar ishga va oʻqishga shoshilib ketayotgan bir paytda men qahva ichib oʻtirgan edim va ertangi rejam haqida chuqur oʻylab koʻrdim
agar siz har kuni kamida yarim soat kitob oʻqisangiz va yangi soʻzlarni kundalik hayotingizda faol qoʻllashga harakat qilsangiz bir necha oydan keyin nutqingiz ancha ravon va boy boʻlib qolishiga shubha qilmayman
bahor fasli kirib kelishi bilan tabiat uygʻonib daraxtlar kurtak chiqara boshlaganda bogʻdagi qushlarning sayrashi va shamolning mayin esishi odamga oʻzgacha ilhom va xotirjamlik bagʻishlaydi
biz oʻtgan yili yozda doʻstlarimiz bilan togʻ etagiga dam olishga borganimizda kechqurun gulxan yoqib atrofida oʻtirdik va bolalik xotiralari hamda kelajakdagi orzularimiz haqida uzoq suhbatlashdik
zamonaviy texnologiyalar kundalik hayotimizni sezilarli darajada osonlashtirgan boʻlsa-da jonli muloqot va tabiat qoʻynida vaqt oʻtkazish insonning ruhiy salomatligi uchun hamon eng muhim omillardan biri boʻlib qolmoqda"""
inputs = tokenizer(f"correct: {text}", return_tensors="pt")
output_ids = model.generate(**inputs, max_new_tokens=256)
prediction = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]
print(prediction)


Kecha ertalab quyosh hali to'g'liq ko'tarilmasdan oldin shahar ko'chalarida odamlar ishga va o'zishga shoshilib ketayotgan bir paytda men qahva ichib o'ztirgan edim va ertangi rejam haqida chuqur o'zylab ko'zrdim: agar siz har kuni kamida yarim soat kitob 


In [7]:
import json
from pathlib import Path

transcripts_dir = Path("data/transcripts")

print(transcripts_dir.glob("*.json"))

<generator object Path.glob at 0x10bd118b0>


In [17]:
from pathlib import Path
import soundfile as sf

audios_dir = Path("../data/raw_audios")  #
wav_files = list(audios_dir.glob("*.wav"))  # or .rglob("*.wav") for subfolders

if not wav_files:
  print("No .wav files found.")
else:
  longest = None
  shortest = None

  for audio in wav_files:
    info = sf.info(audio)  #[cite: 2]
    duration = info.duration  # in seconds[cite: 2]

    if longest is None or duration > longest["duration"]:
      longest = {"path": audio, "duration": duration, "info": info}

    if shortest is None or duration < shortest["duration"]:
      shortest = {"path": audio, "duration": duration, "info": info}

  print(f"Longest:  {longest['path'].name} ({longest['duration']:.2f}s)")
  print(f"Shortest: {shortest['path'].name} ({shortest['duration']:.2f}s)")

Longest:  Pulni maqsad qilganlar kambag‘al bo‘ladi.wav (9894.46s)
Shortest: hayotingiz isrof bo'lishiga sabab (yo'q, telefon emas).wav (845.76s)
